# 01. The 2024 and 2026 junior software pools rebuilt from raw LinkedIn corpora

**Design decisions.**
- 2024 source: the Kaggle dataset "LinkedIn Job Postings (2023-2024)" (`arshkon/linkedin-job-postings`, `postings.csv`, 123,849 rows as cited in the paper), downloaded once to `data/raw/linkedin_2024/` with Kaggle credentials asked for by getpass and stored in `Hiring_Research/.secrets/kaggle.json`.
- 2026 source: the 2026 first-party LinkedIn scrape across 26 US metropolitan areas released with Zenodo record 20321172 (CC BY 4.0, `scraped_postings.parquet`, about 363,000 postings), downloaded once to `data/raw/linkedin_2026/` from the public Zenodo API. Only the raw title and description are used; the record's own model-derived labels are not. The release documentation zip is downloaded too and its codebook printed so the column mapping can be checked. The 2024 and 2026 corpora are therefore the same platform and country; the 2019 Monster USA baseline remains a different source and is reported as such.
- The June 2026 pool of 500 did not survive, only the rated sample of 120. Both pools are rebuilt with the written rule in `src/subsets.py`: software-development title, non-software roles excluded, junior status from a title cue or from the platform's experience-level field where the corpus has one, and never a title with an explicit senior cue. Pool sizes are whatever the rule gives; the paper's text follows the rule.
- Each selected advertisement records its route (`why`: title cue, platform level, or both). The June 2024 sample and the June 2026 subset are compared with the rebuilt pools and the reason for every June advertisement the rule does not keep is listed.
- Column names of the parquet file are picked automatically from candidate lists (`src/fetch.py`); if a required column is not found the cell stops and prints the schema, and the name can be set in SETTINGS.
- Outputs: `data/subsets/junior_2024.csv`, `junior_2026.csv` (gitignored, have text); `reports/subset_2024_summary.json`, `subset_2026_summary.json`, `pool_sizes.csv` (year, route, N), and the June comparisons.


In [ ]:
# === SETTINGS ===
import os
from pathlib import Path
REPO_URL   = "https://github.com/anasbiswas1/hiring-research.git"
DRIVE_ROOT = Path(os.environ.get("HIRING_DRIVE_ROOT", "/content/drive/MyDrive/Hiring_Research"))
REPO_DIR   = DRIVE_ROOT / "hiring-research"
KAGGLE_DATASET = "arshkon/linkedin-job-postings"   # LinkedIn Job Postings (2023-2024)
EXPECTED_ROWS  = 123849                               # as cited in the paper; a different count is reported, not hidden
RAW_2024       = REPO_DIR / "data" / "raw" / "linkedin_2024"
ZENODO_RECORD  = "20321172"                           # AI-era software-engineering job postings: a 2024-2026 panel (CC BY 4.0)
ZENODO_FILE    = "scraped_postings.parquet"           # the 2026 first-party scrape, one row per posting
ZENODO_DOCS    = "swe-labor-release-v1-docs.zip"
RAW_2026       = REPO_DIR / "data" / "raw" / "linkedin_2026"
COL_2026       = {"title": None, "desc": None, "level": None, "date": None, "id": None}   # None = pick automatically


In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import os, sys, shutil, subprocess, importlib
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
CRED = DRIVE_ROOT / ".secrets"
for name, dot in [("git-credentials", ".git-credentials"), ("gitconfig", ".gitconfig")]:
    if (CRED / name).exists():
        shutil.copy(CRED / name, Path.home() / dot)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"])
os.environ["HIRING_DRIVE_ROOT"] = str(DRIVE_ROOT)
os.environ["HIRING_REPO_ROOT"] = str(REPO_DIR)
assert (REPO_DIR / ".git").exists(), f"no git checkout at {REPO_DIR}: run notebook 00 Stage 0 first"
os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()
pull = subprocess.run(["git", "pull", "origin", "main"], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())
print("repo:", REPO_DIR)


## Stage 0. Install

In [ ]:
# === INSTALL ===
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *['kaggle', 'pyarrow', 'pandas']], capture_output=True, text=True)
print("pip -> rc", r.returncode)
print(r.stderr[-600:])


## Stage 1. Kaggle credentials

In [ ]:
# === STAGE 1: Kaggle credentials (asked once, stored in .secrets, never committed) ===
import os, json, getpass
from pathlib import Path
secrets = DRIVE_ROOT / ".secrets"
secrets.mkdir(parents=True, exist_ok=True)
kj = secrets / "kaggle.json"
if not kj.exists():
    user = input("Kaggle username: ").strip()
    key = getpass.getpass("Kaggle API key (from kaggle.com/settings, 'Create New Token'): ").strip()
    kj.write_text(json.dumps({"username": user, "key": key}))
    del key
os.chmod(kj, 0o600)
creds = json.loads(kj.read_text())
os.environ["KAGGLE_USERNAME"] = creds["username"]
os.environ["KAGGLE_KEY"] = creds["key"]
print("kaggle credentials loaded for", creds["username"])


## Stage 2. 2024: download and schema check\nThe cell stops with a clear message if the dataset does not have `title` and `description` columns.

In [ ]:
# === STAGE 2: 2024 download once, then find and check postings.csv ===
import subprocess, sys, shutil
import pandas as pd
RAW_2024.mkdir(parents=True, exist_ok=True)
postings = sorted(RAW_2024.rglob("postings.csv"))
if not postings:
    kaggle_cmd = [shutil.which("kaggle")] if shutil.which("kaggle") else [sys.executable, "-m", "kaggle"]
    r = subprocess.run([*kaggle_cmd, "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(RAW_2024), "--unzip"],
                       capture_output=True, text=True)
    print(r.stdout[-1500:]); print(r.stderr[-1500:])
    postings = sorted(RAW_2024.rglob("postings.csv"))
assert postings, f"postings.csv not found under {RAW_2024}; files present: {[p.name for p in RAW_2024.rglob('*') if p.is_file()][:20]}"
POSTINGS = postings[0]
header = pd.read_csv(POSTINGS, nrows=0).columns.tolist()
print("file:", POSTINGS)
print("columns:", header)
for col in ("title", "description"):
    assert col in header, f"column {col!r} missing: the dataset schema differs from the one the rule was written for; stop here"
LEVEL_COL = "formatted_experience_level" if "formatted_experience_level" in header else None
print("experience-level column:", LEVEL_COL)


## Stage 3. 2024 pool by the written rule, and the comparison with the June sample

In [ ]:
# === STAGE 3: 2024 pool by the written rule, and the comparison with the June sample ===
import importlib, json
import pandas as pd
import ads, subsets
importlib.reload(ads); importlib.reload(subsets)

def compare_with_june(pool, june):
    in_pool = june.ad_id.isin(set(pool.ad_id))
    key_pool = set((pool.title.str.lower() + "|" + pool.desc.str[:200].str.lower()))   # looser match for text cleaned differently in June
    loose = (june.title.str.lower() + "|" + june.desc.str[:200].str.lower()).isin(key_pool)
    reasons = []
    for t, hit in zip(june.title, in_pool | loose):
        if hit: reasons.append("in pool")
        elif not subsets.is_software(t): reasons.append("not a software-development title")
        elif subsets.has_senior_cue(t): reasons.append("senior cue in title")
        elif not subsets.has_junior_cue(t): reasons.append("no title cue and platform level not junior (or text differs)")
        else: reasons.append("title passes the rule but the advertisement text was not matched")
    return june.assign(in_rebuilt_pool=(in_pool | loose), reason=reasons)

def pool_summary(raw, sel, pool, level_col, source_file, expected_rows=None):
    out = {"source_file": source_file, "rows": int(len(raw)), "expected_rows": expected_rows,
           "selection_counts": subsets.selection_counts(raw, "title", level_col), "pool_size": int(len(pool)),
           "route": pool.why.value_counts().to_dict(), "level": pool.level.fillna("").astype(str).value_counts().head(10).to_dict(),
           "top_titles": pool.title.value_counts().head(25).to_dict()}
    return out

usecols = [c for c in ["job_id", "company_name", "title", "description", LEVEL_COL, "listed_time", "location"] if c and c in header]
raw24 = pd.read_csv(POSTINGS, usecols=usecols, dtype=str)
print(f"2024 rows: {len(raw24)} (paper cites {EXPECTED_ROWS})")
if len(raw24) != EXPECTED_ROWS:
    print("NOTE: row count differs from the paper's figure; report the actual count")
sel24 = subsets.select_junior_software(raw24, "title", LEVEL_COL)
pool24 = ads.make_subset(sel24, 2024, "LinkedIn", title_col="title", desc_col="description")
meta = sel24.assign(ad_id=[ads.ad_id(t, d) for t, d in zip(sel24.title, sel24.description)]).drop_duplicates("ad_id")
keep = ["ad_id", "why"] + ([LEVEL_COL] if LEVEL_COL else []) + (["job_id"] if "job_id" in meta else [])
pool24 = pool24.merge(meta[keep], on="ad_id", how="left").rename(columns={LEVEL_COL: "level"} if LEVEL_COL else {})
if "level" not in pool24: pool24["level"] = ""
pool24.to_csv(REPO_DIR / "data" / "subsets" / "junior_2024.csv", index=False)

june24 = pd.read_csv(REPO_DIR / "data" / "subsets" / "june_2024_sample.csv", dtype={"ad_id": str})
cmp24 = compare_with_june(pool24, june24)
summary24 = pool_summary(raw24, sel24, pool24, LEVEL_COL, str(POSTINGS.relative_to(REPO_DIR)), EXPECTED_ROWS)
summary24.update({"june_sample_in_pool": int(cmp24.in_rebuilt_pool.sum()), "june_sample_n": int(len(june24)),
                  "june_sample_not_in_pool_reasons": cmp24[~cmp24.in_rebuilt_pool].reason.value_counts().to_dict()})
ads.write_json(REPO_DIR / "reports" / "subset_2024_summary.json", summary24)
cmp24[["ad_id", "title", "in_rebuilt_pool", "reason"]].to_csv(REPO_DIR / "reports" / "june_2024_sample_vs_pool.csv", index=False)
print(json.dumps({k: v for k, v in summary24.items() if k not in ("top_titles", "level")}, indent=2))
print("2024 pool saved:", len(pool24), "advertisements; route:", summary24["route"])


## Stage 4. 2026: the Zenodo scrape (download once, codebook shown)

In [ ]:
# === STAGE 4: 2026 scrape from the Zenodo record, once; codebook printed ===
import importlib, zipfile
import fetch
importlib.reload(fetch)
RAW_2026.mkdir(parents=True, exist_ok=True)
files = {f["key"]: f for f in fetch.zenodo_files(ZENODO_RECORD)}
print("record files:", {k: f"{v['size'] / 1e6:.0f} MB" for k, v in files.items()})
assert ZENODO_FILE in files, f"{ZENODO_FILE} not in the record; set ZENODO_FILE to one of {list(files)}"
PARQUET_2026 = fetch.download(files[ZENODO_FILE]["url"], RAW_2026 / ZENODO_FILE, files[ZENODO_FILE]["size"])
if ZENODO_DOCS in files:
    docs = fetch.download(files[ZENODO_DOCS]["url"], RAW_2026 / ZENODO_DOCS, files[ZENODO_DOCS]["size"])
    with zipfile.ZipFile(docs) as z:
        names = z.namelist()
        print("docs:", names)
        for n in names:
            if n.lower().endswith("codebook.md"):
                text = z.read(n).decode("utf-8", errors="ignore")
                i = text.lower().find("scraped_postings")
                print(text[max(0, i - 200): i + 3500] if i >= 0 else text[:3500])
                break
print("columns:", fetch.parquet_columns(PARQUET_2026))


## Stage 5. 2026 pool by the same rule, comparison with the June 2026 subset, pool sizes by route

In [ ]:
# === STAGE 5: 2026 pool by the same rule, comparison with the June 2026 subset, pool sizes ===
import importlib, json
import pandas as pd
import ads, subsets, fetch
cols = fetch.parquet_columns(PARQUET_2026)
T = COL_2026["title"] or fetch.pick_column(cols, fetch.TITLE_CANDIDATES, what="title")
D = COL_2026["desc"] or fetch.pick_column(cols, fetch.DESC_CANDIDATES, what="description")
L = COL_2026["level"] or fetch.pick_column(cols, fetch.LEVEL_CANDIDATES, required=False)
DT = COL_2026["date"] or fetch.pick_column(cols, fetch.DATE_CANDIDATES, required=False)
ID = COL_2026["id"] or fetch.pick_column(cols, fetch.ID_CANDIDATES, required=False)
print("using columns:", {"title": T, "desc": D, "level": L, "date": DT, "id": ID})
use = [c for c in (ID, T, D, L, DT) if c]
raw26 = pd.read_parquet(PARQUET_2026, columns=use)
raw26 = raw26.rename(columns={T: "title", D: "description"})
if L: raw26 = raw26.rename(columns={L: "level_raw"})
print(f"2026 rows: {len(raw26)}")
if DT:
    d = pd.to_datetime(raw26[DT], errors="coerce", utc=True)
    print("posting dates:", d.min(), "to", d.max(), f"({d.notna().mean():.0%} parseable)")
    DATE_RANGE = [str(d.min()), str(d.max())]
else:
    DATE_RANGE = None
if "level_raw" in raw26:
    print("level values:", raw26.level_raw.astype(str).value_counts().head(10).to_dict())
sel26 = subsets.select_junior_software(raw26, "title", "level_raw" if "level_raw" in raw26 else None)
pool26 = ads.make_subset(sel26, 2026, "LinkedIn", title_col="title", desc_col="description")
meta26 = sel26.assign(ad_id=[ads.ad_id(t, d) for t, d in zip(sel26.title, sel26.description)]).drop_duplicates("ad_id")
keep = ["ad_id", "why"] + (["level_raw"] if "level_raw" in meta26 else []) + ([ID] if ID else [])
pool26 = pool26.merge(meta26[keep], on="ad_id", how="left").rename(columns={"level_raw": "level", ID: "source_id"} if ID else {"level_raw": "level"})
if "level" not in pool26: pool26["level"] = ""
pool26.to_csv(REPO_DIR / "data" / "subsets" / "junior_2026.csv", index=False)

june26 = pd.read_csv(REPO_DIR / "data" / "subsets" / "june_2026_subset.csv", dtype={"ad_id": str})
cmp26 = compare_with_june(pool26, june26)
summary26 = pool_summary(raw26, sel26, pool26, "level_raw" if "level_raw" in raw26 else None, f"zenodo:{ZENODO_RECORD}/{ZENODO_FILE}")
summary26.update({"columns_used": {"title": T, "desc": D, "level": L, "date": DT, "id": ID}, "date_range": DATE_RANGE,
                  "june_subset_in_pool": int(cmp26.in_rebuilt_pool.sum()), "june_subset_n": int(len(june26)),
                  "june_subset_not_in_pool_reasons": cmp26[~cmp26.in_rebuilt_pool].reason.value_counts().to_dict()})
ads.write_json(REPO_DIR / "reports" / "subset_2026_summary.json", summary26)
cmp26[["ad_id", "title", "in_rebuilt_pool", "reason"]].to_csv(REPO_DIR / "reports" / "june_2026_subset_vs_pool.csv", index=False)
print(json.dumps({k: v for k, v in summary26.items() if k not in ("top_titles", "level")}, indent=2))

import results
pool19 = pd.read_csv(REPO_DIR / "data" / "subsets" / "junior_2019.csv", dtype={"ad_id": str})
sizes = pd.concat([p.assign(route=p.why.map(results.route_of)).groupby(["year", "route"]).size().rename("N").reset_index()
                   for p in (pool19, pool24, pool26)])
sizes.to_csv(REPO_DIR / "reports" / "pool_sizes.csv", index=False)
print(sizes.to_string(index=False))


## End of unit: commit

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "symbolic-ref", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", 'nb01: 2024 pool from the LinkedIn 2023-2024 postings and 2026 pool from the Zenodo 20321172 LinkedIn scrape, both by the written rule; reports/subset_2024_summary.json, subset_2026_summary.json, pool_sizes.csv, June comparisons'], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)
